# PPMI Biospecimen Data Exploration

## Purpose

Explore the structure and content of the PPMI biospecimen data before
defining any feature-selection or preprocessing strategy.

This notebook focuses on understanding:

- the structure of the biospecimen dataset,
- the available biospecimen `TYPE`s,
- the tests (`TESTNAME`) available within each type,
- observation and participant counts,
- measurement units,
- clinical events and longitudinal structure,
- missingness and repeated measurements.

No modelling or preprocessing decisions are made in this notebook.

In [2]:
%load_ext autoreload
%autoreload 2

In [9]:
import pandas as pd
from pathlib import Path

## Load biospecimen data

The biospecimen data are stored in long format, where each row represents
a biospecimen test result for a participant at a particular clinical event.

The project root is located dynamically by searching for `pyproject.toml`.

In [10]:
from pathlib import Path


PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent



In [12]:
PROJECT_ROOT = Path.cwd()

while not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

current_biospecimen = pd.read_csv(
    PROJECT_ROOT
    / "data/raw/ppmi/Current_Biospecimen_Analysis_Results_28Jul2026.csv"
)

C:\Users\eademola\AppData\Local\Temp\ipykernel_38168\2262976388.py:6: DtypeWarning: Columns (6,7) have mixed types. Specify dtype option on import or set low_memory=False.
  current_biospecimen = pd.read_csv(


## 1. Dataset overview

First, inspect the overall dimensions and column structure of the
biospecimen dataset.

In [13]:
print(f"Number of rows: {len(current_biospecimen):,}")
print(f"Number of columns: {current_biospecimen.shape[1]}")

Number of rows: 1,156,000
Number of columns: 13


In [14]:
current_biospecimen.columns

Index(['PATNO', 'SEX', 'COHORT', 'CLINICAL_EVENT', 'TYPE', 'TESTNAME',
       'TESTVALUE', 'UNITS', 'RUNDATE', 'PROJECTID', 'PI_NAME',
       'PI_INSTITUTION', 'update_stamp'],
      dtype='object')

In [15]:
current_biospecimen.head()

,PATNO,SEX,COHORT,CLINICAL_EVENT,TYPE,TESTNAME,TESTVALUE,UNITS,RUNDATE,PROJECTID,PI_NAME,PI_INSTITUTION,update_stamp
0,3000,Female,Control,SC,DNA,ApoE Genotype,e3/e3,NaN,2012-09-27,104,Andrew Singleton,National Institutes of Aging,2014-11-03 11:58:57
1,3001,Male,PD,SC,DNA,ApoE Genotype,e3/e3,NaN,2012-09-27,104,Andrew Singleton,National Institutes of Aging,2014-11-03 11:58:57
2,3002,Female,PD,SC,DNA,ApoE Genotype,e3/e3,NaN,2012-09-27,104,Andrew Singleton,National Institutes of Aging,2014-11-03 11:58:57
3,3003,Female,PD,SC,DNA,ApoE Genotype,e4/e3,NaN,2012-09-27,104,Andrew Singleton,National Institutes of Aging,2014-11-03 11:58:57
4,3004,Male,Control,SC,DNA,ApoE Genotype,e3/e2,NaN,2012-09-27,104,Andrew Singleton,National Institutes of Aging,2014-11-03 11:58:57


## 2. Column data types and missingness

Inspect the data type of each column and identify missing values.

At this stage, this is purely descriptive. In particular, `TESTVALUE`
may require additional investigation because biospecimen results can
potentially be represented in different formats.

In [16]:
biospecimen_dtypes = (
    current_biospecimen
    .dtypes
    .rename("dtype")
    .reset_index()
    .rename(columns={"index": "column"})
)

biospecimen_dtypes

,column,dtype
0,PATNO,int64
1,SEX,object
2,COHORT,object
3,CLINICAL_EVENT,object
4,TYPE,object
5,TESTNAME,object
6,TESTVALUE,object
7,UNITS,object
8,RUNDATE,object
9,PROJECTID,int64


In [17]:
biospecimen_missingness = (
    current_biospecimen
    .isna()
    .sum()
    .rename("n_missing")
    .reset_index()
    .rename(columns={"index": "column"})
)

biospecimen_missingness["missing_percentage"] = (
    biospecimen_missingness["n_missing"]
    / len(current_biospecimen)
    * 100
)

biospecimen_missingness.sort_values(
    "missing_percentage",
    ascending=False,
)

,column,n_missing,missing_percentage
7,UNITS,135551,11.725865
6,TESTVALUE,67476,5.837024
2,COHORT,0,0.000000
1,SEX,0,0.000000
0,PATNO,0,0.000000
4,TYPE,0,0.000000
3,CLINICAL_EVENT,0,0.000000
5,TESTNAME,0,0.000000
8,RUNDATE,0,0.000000
9,PROJECTID,0,0.000000


## 3. Biospecimen types

`TYPE` appears to define broad categories of biospecimen measurements.

First, inspect the unique values and their frequencies.

Both the number of observations and the number of unique participants
are reported because the data are in long format. A large number of
observations does not necessarily correspond to a large number of
participants.

In [18]:
biospecimen_type_summary = (
    current_biospecimen
    .groupby("TYPE", dropna=False)
    .agg(
        n_observations=("PATNO", "size"),
        n_participants=("PATNO", "nunique"),
    )
    .reset_index()
    .sort_values(
        "n_observations",
        ascending=False,
    )
)

biospecimen_type_summary

,TYPE,n_observations,n_participants
6,Plasma,731800,1946
2,Cerebrospinal Fluid,147826,2470
3,DNA,131784,1205
1,Cell Line,51881,129
9,Serum,35558,1434
17,miRNA,26285,612
12,Tissue Frozen,10236,120
4,PBMC,6018,121
15,Urine,4827,1186
13,Tissue Homogenate,4152,362


In [19]:
current_biospecimen["TYPE"].unique()

array(['DNA', 'miRNA', 'Plasma', 'Serum', 'RNA', 'Cerebrospinal Fluid',
       'CSF', 'Cell Line', 'Whole Blood', 'SERUM', 'Urine', 'URINE',
       'PLASMA', 'Tissue Frozen', 'Tissue Formalin', 'PBMC',
       'Tissue Fixed', 'Tissue Homogenate'], dtype=object)

## 4. Tests available within each biospecimen type

Next, inspect the structure of `TESTNAME` within each `TYPE`.

This will help identify the different biomarkers or laboratory tests
represented in each biospecimen category.

In [20]:
biospecimen_test_summary = (
    current_biospecimen
    .groupby(["TYPE", "TESTNAME"], dropna=False)
    .agg(
        n_observations=("PATNO", "size"),
        n_participants=("PATNO", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["TYPE", "n_participants"],
        ascending=[True, False],
    )
)

biospecimen_test_summary

,TYPE,TESTNAME,n_observations,n_participants
37,CSF,pTau,86,47
39,CSF,tTau,86,47
1,CSF,ABeta 1-42,37,32
13,CSF,CSF Alpha-synuclein,37,32
14,CSF,CSF Hemoglobin,37,32
...,...,...,...,...
9100,miRNA,HNF4A,400,200
9110,miRNA,PTBP1,400,200
9119,miRNA,SOD2,400,200
9124,miRNA,WLS,400,200


In [22]:
for biospecimen_type, group in current_biospecimen.groupby(
    "TYPE",
    dropna=False,
):
    tests = sorted(group["TESTNAME"].dropna().unique())

    print(f"\n=== {biospecimen_type} ===")
    print(f"Number of unique tests: {len(tests)}")

    for test in tests:
        print(f"  - {test}")


=== CSF ===
Number of unique tests: 40
  - ABeta
  - ABeta 1-42
  - ABeta raw
  - ABeta40
  - ABeta42
  - Acetylputrescine
  - Acetylspermidine
  - Aceytylspermine
  - CATB
  - CATD
  - CATF
  - CATL
  - CATS
  - CSF Alpha-synuclein
  - CSF Hemoglobin
  - CSF LRRK2
  - Diacetylspermidine
  - Diacetylspermine
  - GBA
  - GCase activity
  - GM2A
  - GPNMB
  - Glucosylsphingosine
  - LAB_QC_FLAG
  - LAMP1
  - LAMP2
  - LC3
  - LRRK2 CSF
  - LRRK2 CSF_batch_corr
  - LRRK2 CSF_raw
  - Progranulin
  - Putrescine
  - Spermidine
  - Spermine
  - Ubiquitin
  - eMTBR-TAU243
  - pS65 Ubiquitin
  - pTau
  - pTau181
  - tTau

=== Cell Line ===
Number of unique tests: 5215
  - %LPS phagocytosis LPS (rep 1)
  - %LPS phagocytosis LPS (rep 2)
  - %LPS phagocytosis LPS+MLi-2 (rep 1)
  - %LPS phagocytosis LPS+NA (rep 1)
  - %LPS phagocytosis LPS+NA (rep 2)
  - %LPS phagocytosis MLi-2 (rep 2)
  - %LPS phagocytosis unstim (rep 1)
  - %LPS phagocytosis unstim (rep 2)
  - ?-syn/actinH2O2
  - ?-syn/actinH2O2

## 5. Frequency of individual tests

Examine how frequently each test occurs across the dataset.

Both observation counts and participant counts are retained because
multiple measurements may exist for the same participant.

In [ ]:
biospecimen_test_frequency = (
    current_biospecimen
    .groupby(["TYPE", "TESTNAME"], dropna=False)
    .agg(
        n_observations=("PATNO", "size"),
        n_participants=("PATNO", "nunique"),
    )
    .reset_index()
    .sort_values(
        "n_participants",
        ascending=False,
    )
)

biospecimen_test_frequency

## 6. Measurement units

Check which units are associated with each test.

Ideally, a given test should have a consistent unit. Tests with multiple
units require further investigation before their numerical values can
be compared directly.

In [23]:
biospecimen_unit_summary = (
    current_biospecimen
    .groupby(
        ["TYPE", "TESTNAME"],
        dropna=False,
    )["UNITS"]
    .agg(
        n_unique_units="nunique",
        units=lambda values: sorted(
            values.dropna().astype(str).unique()
        ),
    )
    .reset_index()
)

biospecimen_unit_summary

,TYPE,TESTNAME,n_unique_units,units
0,CSF,ABeta,1,[pg/mL]
1,CSF,ABeta 1-42,1,[pg/mL]
2,CSF,ABeta raw,1,[pg/mL]
3,CSF,ABeta40,1,[pg/mL]
4,CSF,ABeta42,1,[pg/mL]
...,...,...,...,...
9122,miRNA,UBE2K (rep 1),1,[Ct]
9123,miRNA,UBE2K (rep 2),1,[Ct]
9124,miRNA,WLS,2,"[Avg CT, SD]"
9125,miRNA,ZNF160,2,"[Avg CT, SD]"


In [24]:
biospecimen_tests_multiple_units = (
    biospecimen_unit_summary[
        biospecimen_unit_summary["n_unique_units"] > 1
    ]
    .copy()
)

biospecimen_tests_multiple_units

,TYPE,TESTNAME,n_unique_units,units
36,CSF,pS65 Ubiquitin,2,"[average (fM), stdev]"
295,Cell Line,Ferritin Light Chain levels (normalized to bet...,2,"[Average (FTL/Actin Arb. Un.), Stdev]"
296,Cell Line,FerroOrange Live Cell Imaging intensity,2,"[Average (Intensity/cell), Stdev]"
340,Cell Line,GBA1/Actin IPSC's,2,"[Average (GBA1/Actin Arb. Un.), Stdev]"
529,Cell Line,ICC-Day 25-% FoxA2 of TH,2,"[Percent, Stdev]"
...,...,...,...,...
9100,miRNA,HNF4A,2,"[Avg CT, SD]"
9110,miRNA,PTBP1,2,"[Avg CT, SD]"
9119,miRNA,SOD2,2,"[Avg CT, SD]"
9124,miRNA,WLS,2,"[Avg CT, SD]"


## 7. Clinical events

Inspect `CLINICAL_EVENT` to understand when biospecimen measurements
were collected.

Because the prediction task uses baseline information to predict
subsequent phenoconversion, it is important to understand the
longitudinal structure of the biospecimen data before selecting
baseline measurements.

In [25]:
biospecimen_event_summary = (
    current_biospecimen
    .groupby("CLINICAL_EVENT", dropna=False)
    .agg(
        n_observations=("PATNO", "size"),
        n_participants=("PATNO", "nunique"),
    )
    .reset_index()
    .sort_values(
        "n_participants",
        ascending=False,
    )
)

biospecimen_event_summary

,CLINICAL_EVENT,n_observations,n_participants
0,BL,306179,2853
9,V04,126448,1293
11,V06,77012,1160
7,V02,64328,1127
2,SC,104944,964
13,V08,103254,817
15,V10,11474,623
17,V12,24983,610
20,V15,16993,255
19,V14,13365,240


In [26]:
current_biospecimen["CLINICAL_EVENT"].unique()

array(['SC', 'BL', 'V02', 'V01', 'V04', 'V03', 'ST', 'V06', 'PW', 'U01',
       'V08', 'V07', 'V05', 'V10', 'V09', 'V11', 'U02', 'V12', 'V14',
       'V13', 'V16', 'V15', 'V17', 'V18', 'V19', 'V20'], dtype=object)

## 8. Test availability across clinical events

Examine how individual biospecimen tests are distributed across
clinical events.

This provides an initial view of whether biomarkers are collected
only at baseline or repeatedly over follow-up.

In [27]:
biospecimen_test_event_summary = (
    current_biospecimen
    .groupby(
        ["TYPE", "TESTNAME", "CLINICAL_EVENT"],
        dropna=False,
    )
    .agg(
        n_observations=("PATNO", "size"),
        n_participants=("PATNO", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["TYPE", "TESTNAME", "CLINICAL_EVENT"],
    )
)

biospecimen_test_event_summary

,TYPE,TESTNAME,CLINICAL_EVENT,n_observations,n_participants
0,CSF,ABeta,BL,14,14
1,CSF,ABeta,V04,11,11
2,CSF,ABeta,V06,9,9
3,CSF,ABeta,V08,10,10
4,CSF,ABeta,V10,3,3
...,...,...,...,...,...
38758,miRNA,UBE2K (rep 2),V07,1,1
38759,miRNA,UBE2K (rep 2),V08,173,173
38760,miRNA,WLS,BL,400,200
38761,miRNA,ZNF160,BL,400,200


## 9. Repeated measurements

Because the dataset is longitudinal and in long format, determine
whether participants can have multiple observations for the same
test and clinical event.

This is descriptive exploration only; no rule for resolving
duplicates or repeated measurements is defined here.

In [28]:
biospecimen_repeated_measurements = (
    current_biospecimen
    .groupby(
        ["PATNO", "CLINICAL_EVENT", "TYPE", "TESTNAME"],
        dropna=False,
    )
    .size()
    .rename("n_measurements")
    .reset_index()
)

biospecimen_repeated_measurements[
    biospecimen_repeated_measurements["n_measurements"] > 1
].sort_values(
    "n_measurements",
    ascending=False,
)

,PATNO,CLINICAL_EVENT,TYPE,TESTNAME,n_measurements
444572,4105,V06,Cell Line,ACTB ipsc,19
444568,4105,V06,Cell Line,ACTB diff2 day 30,10
211176,3445,V07,Cell Line,ACTB ipsc,10
444567,4105,V06,Cell Line,ACTB diff1 day60,10
444571,4105,V06,Cell Line,ACTB diff3 day 60,10
...,...,...,...,...,...
469,3000,V12,Plasma,pS65 Ubiquitin,2
475,3000,V15,Plasma,pS65 Ubiquitin,2
476,3000,V17,Plasma,pS65 Ubiquitin,2
487,3001,BL,Cerebrospinal Fluid,ABeta40,2


## 10. Test value representation

Inspect the representation of `TESTVALUE`.

The goal at this stage is to determine whether test results are
consistently numeric or whether some tests contain non-numeric values,
qualifiers, or other representations that will require interpretation.

In [29]:
current_biospecimen["TESTVALUE"].head(20)

0     e3/e3
1     e3/e3
2     e3/e3
3     e4/e3
4     e3/e2
5     e3/e2
6     e4/e3
7     e3/e2
8     e3/e3
9     e4/e3
10    e3/e3
11    e3/e3
12    e4/e3
13    e3/e3
14    e3/e3
15    e4/e3
16    e3/e3
17    e3/e2
18    e3/e2
19    e3/e3
Name: TESTVALUE, dtype: object

In [30]:
current_biospecimen["TESTVALUE"].map(type).value_counts()

TESTVALUE
<class 'str'>      629772
<class 'float'>    526228
Name: count, dtype: int64

In [31]:
biospecimen_testvalue_numeric = pd.to_numeric(
    current_biospecimen["TESTVALUE"],
    errors="coerce",
)

print(
    "Numeric TESTVALUE:",
    biospecimen_testvalue_numeric.notna().sum(),
)

print(
    "Non-numeric TESTVALUE:",
    biospecimen_testvalue_numeric.isna().sum(),
)

Numeric TESTVALUE: 945537
Non-numeric TESTVALUE: 210463


## 11. Biospecimen test inventory

Create a compact inventory of the biospecimen tests, including their
frequency, participant coverage, and number of observed units.

This table serves as a reference for subsequent feature-selection
exploration.

In [33]:
biospecimen_test_inventory = (
    current_biospecimen
    .groupby(
        ["TYPE", "TESTNAME"],
        dropna=False,
    )
    .agg(
        n_observations=("PATNO", "size"),
        n_participants=("PATNO", "nunique"),
        n_clinical_events=("CLINICAL_EVENT", "nunique"),
        n_units=("UNITS", "nunique"),
    )
    .reset_index()
    .sort_values(
        ["TYPE", "n_participants"],
        ascending=[True, False],
    )
)

biospecimen_test_inventory

,TYPE,TESTNAME,n_observations,n_participants,n_clinical_events,n_units
37,CSF,pTau,86,47,6,1
39,CSF,tTau,86,47,6,1
1,CSF,ABeta 1-42,37,32,2,1
13,CSF,CSF Alpha-synuclein,37,32,2,1
14,CSF,CSF Hemoglobin,37,32,2,1
...,...,...,...,...,...,...
9100,miRNA,HNF4A,400,200,1,2
9110,miRNA,PTBP1,400,200,1,2
9119,miRNA,SOD2,400,200,1,2
9124,miRNA,WLS,400,200,1,2
